<a href="https://colab.research.google.com/github/mingo002/graph-ml-peptide-binding/blob/main/wash_pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

import pandas as pd

from pathlib import Path


In [5]:
df_train = pd.read_csv("pep_nolog_sum_cf20_cf200_train.csv")
df_test = pd.read_csv("pep_nolog_sum_cf20_cf200_test.csv")

In [6]:
VHSE8 = {
    "A": [ 0.15, -1.11, -1.35, -0.92,  0.02, -0.91,  0.36, -0.48],
    "R": [-1.47,  1.45,  1.24,  1.27,  1.55,  1.47,  1.30,  0.83],
    "N": [-0.99,  0.00, -0.37,  0.69, -0.55,  0.85,  0.73, -0.80],
    "D": [-1.15,  0.67, -0.41, -0.01, -2.68,  1.31,  0.03,  0.56],
    "C": [ 0.18, -1.67, -0.46, -0.21,  0.00,  1.20, -1.61, -0.19],
    "Q": [-0.96,  0.12,  0.18,  0.16,  0.09,  0.42, -0.20, -0.41],
    "E": [-1.18,  0.40,  0.10,  0.36, -2.16, -0.17,  0.91,  0.02],
    "G": [-0.20, -1.53, -2.63,  2.28, -0.53, -1.18,  2.01, -1.34],
    "H": [-0.43, -0.25,  0.37,  0.19,  0.51,  1.28,  0.93,  0.65],
    "I": [ 1.27, -0.14,  0.30, -1.80,  0.30, -1.61, -0.16, -0.13],
    "L": [ 1.36,  0.07,  0.26, -0.80,  0.22, -1.37,  0.08, -0.62],
    "K": [-1.17,  0.70,  0.70,  0.80,  1.64,  0.67,  1.63,  0.13],
    "M": [ 1.01, -0.53,  0.43,  0.00,  0.23,  0.10, -0.86, -0.68],
    "F": [ 1.52,  0.61,  0.96, -0.16,  0.25,  0.28, -1.33, -0.20],
    "P": [ 0.22, -0.17, -0.50,  0.05, -0.01, -1.34, -0.19,  3.56],
    "S": [-0.67, -0.86, -1.07, -0.41, -0.32,  0.27, -0.64,  0.11],
    "T": [-0.34, -0.51, -0.55, -1.06,  0.01, -0.01, -0.79,  0.39],
    "W": [ 1.50,  2.06,  1.79,  0.75,  0.75, -0.13, -1.06, -0.85],
    "Y": [ 0.61,  1.60,  1.17,  0.73,  0.53,  0.25, -0.96, -0.52],
    "V": [ 0.76, -0.92,  0.17, -1.91,  0.22, -1.40, -0.24, -0.03],
}

AA_SET = set(VHSE8.keys())

In [7]:
# Kyte-Doolittle hydrophobicity scale (common + simple)
KD_HYDRO = {
    "A": 1.8,  "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5,
    "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5,
    "L": 3.8,  "K": -3.9, "M": 1.9,  "F": 2.8,  "P": -1.6,
    "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2
}

AROMATIC = set("FWY")
BASIC = set("KRH")
ACIDIC = set("DE")
COORD = set("HCDEY")  # useful for metal/surface interactions
POLAR = set("STNQHDEKR")
HYDROPHOBIC = set("AILMFWVYC")  # rough

def encode_vhse8_flat(seq: str) -> np.ndarray:
    """Return flattened VHSE8 (len(seq)*8,)"""
    seq = seq.strip().upper()
    arr = np.zeros((len(seq), 8), dtype=np.float32)
    for i, aa in enumerate(seq):
        if aa not in VHSE8:
            # unknown AA -> zeros (or raise error)
            arr[i] = 0.0
        else:
            arr[i] = np.array(VHSE8[aa], dtype=np.float32)
    return arr.reshape(-1)

def physchem_features(seq: str) -> dict:
    """Simple peptide-level physicochemical features."""
    seq = seq.strip().upper()
    L = len(seq)
    aa_counts = {aa: seq.count(aa) for aa in AA_SET}

    # Sidechain net charge proxy at ~pH 7 (simple):
    # K,R ~ +1; D,E ~ -1; H ~ +0.1 (partial)
    net_charge = aa_counts["K"] + aa_counts["R"] - aa_counts["D"] - aa_counts["E"] + 0.1*aa_counts["H"]

    hydro_vals = [KD_HYDRO.get(aa, 0.0) for aa in seq]
    mean_hydro = float(np.mean(hydro_vals))
    std_hydro = float(np.std(hydro_vals))

    aromatic_count = sum(aa in AROMATIC for aa in seq)
    basic_count = sum(aa in BASIC for aa in seq)
    acidic_count = sum(aa in ACIDIC for aa in seq)
    coord_count = sum(aa in COORD for aa in seq)
    polar_frac = sum(aa in POLAR for aa in seq) / L
    hydrophobic_frac = sum(aa in HYDROPHOBIC for aa in seq) / L

    # Position-specific flags (optional but useful): does this position contain a coordinating residue?
    coord_pos_flags = {f"coord_pos_{i+1}": int(seq[i] in COORD) for i in range(L)}

    feats = {
        "len": L,
        "net_charge_sidechain": float(net_charge),
        "mean_hydro_kd": mean_hydro,
        "std_hydro_kd": std_hydro,
        "aromatic_count": float(aromatic_count),
        "basic_count": float(basic_count),
        "acidic_count": float(acidic_count),
        "coord_count": float(coord_count),
        "polar_frac": float(polar_frac),
        "hydrophobic_frac": float(hydrophobic_frac),
        "has_H": float(aa_counts["H"] > 0),
        "has_C": float(aa_counts["C"] > 0),
        "has_D": float(aa_counts["D"] > 0),
        "has_E": float(aa_counts["E"] > 0),
        "has_Y": float(aa_counts["Y"] > 0),
    }
    feats.update(coord_pos_flags)
    return feats

In [8]:
# --- TRAIN SET--> VHSE8 flat features (12*8 = 96 columns)
vhse_mat = np.vstack(df_train["peptide"].apply(encode_vhse8_flat).values)
vhse_cols = [f"vhse_pos{pos+1}_d{d+1}" for pos in range(12) for d in range(8)]
vhse_df_train = pd.DataFrame(vhse_mat, columns=vhse_cols, index=df_train.index)

# --- Physchem features ---
phys_df_train = pd.DataFrame(df_train["peptide"].apply(physchem_features).tolist(), index=df_train.index)

# Combine (KEEP peptide_id & peptide for tracking; do NOT treat them as model inputs)
features_df_train = pd.concat(
    [df_train[["pep_ID", "peptide", "total", "wash1", "wash2", "wash3", "wash4"]], vhse_df_train, phys_df_train],
    axis=1
)

features_df_train.head()

,pep_ID,peptide,total,wash1,wash2,wash3,wash4,vhse_pos1_d1,vhse_pos1_d2,vhse_pos1_d3,...,coord_pos_3,coord_pos_4,coord_pos_5,coord_pos_6,coord_pos_7,coord_pos_8,coord_pos_9,coord_pos_10,coord_pos_11,coord_pos_12
0,0,AAAAESLPQASR,203.0,0.743842,0.068966,0.000000,0.187192,0.15,-1.11,-1.35,...,0,0,1,0,0,0,0,0,0,0
1,10,AAAANENWHQKF,829.0,0.580217,0.148372,0.002413,0.268999,0.15,-1.11,-1.35,...,0,0,0,1,0,0,1,0,0,0
2,11,AAAANENWHQNF,1296.0,0.559414,0.137346,0.002315,0.300926,0.15,-1.11,-1.35,...,0,0,0,1,0,0,1,0,0,0
3,23,AAADTLFRWHRM,275.0,0.545455,0.061818,0.003636,0.389091,0.15,-1.11,-1.35,...,0,1,0,0,0,0,0,1,0,0
4,41,AAAGYSLVTMHS,6329.0,0.589193,0.094012,0.002528,0.314268,0.15,-1.11,-1.35,...,0,0,1,0,0,0,0,0,1,0


In [9]:

# --- TEST SET--> VHSE8 flat features (12*8 = 96 columns)
vhse_mat = np.vstack(df_test["peptide"].apply(encode_vhse8_flat).values)
vhse_cols = [f"vhse_pos{pos+1}_d{d+1}" for pos in range(12) for d in range(8)]
vhse_df_test = pd.DataFrame(vhse_mat, columns=vhse_cols, index=df_test.index)

# --- Physchem features ---
phys_df_test = pd.DataFrame(df_test["peptide"].apply(physchem_features).tolist(), index=df_test.index)

# Combine (KEEP peptide_id & peptide for tracking; do NOT treat them as model inputs)
features_df_test = pd.concat(
    [df_test[["pep_ID", "peptide", "total", "wash1", "wash2", "wash3", "wash4"]], vhse_df_test, phys_df_test],
    axis=1
)

features_df_test.head()


,pep_ID,peptide,total,wash1,wash2,wash3,wash4,vhse_pos1_d1,vhse_pos1_d2,vhse_pos1_d3,...,coord_pos_3,coord_pos_4,coord_pos_5,coord_pos_6,coord_pos_7,coord_pos_8,coord_pos_9,coord_pos_10,coord_pos_11,coord_pos_12
0,15519081,QVEVFYATARGH,255,0.588235,0.141176,0.003922,0.266667,-0.96,0.12,0.18,...,1,0,0,1,0,0,0,0,0,1
1,18128100,SRDSSNSMSMGV,910,0.641758,0.098901,0.003297,0.256044,-0.67,-0.86,-1.07,...,1,0,0,0,0,0,0,0,0,0
2,19785195,TMQGNLVRSLQR,637,0.602826,0.105181,0.001570,0.290424,-0.34,-0.51,-0.55,...,0,0,0,0,0,0,0,0,0,0
3,11444486,LRFADVVFSQMD,209,0.483254,0.157895,0.009569,0.349282,1.36,0.07,0.26,...,0,0,1,0,0,0,0,0,0,1
4,153407,ADVDEQAVRKHP,1084,0.634686,0.095941,0.002768,0.266605,0.15,-1.11,-1.35,...,0,1,1,0,0,0,0,0,1,0


In [10]:
import torch
from transformers import AutoTokenizer, AutoModel

def esm2_pooled_embeddings(
    sequences,
    model_name="facebook/esm2_t6_8M_UR50D",  # small & fast
    batch_size=256,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()

    all_embs = []
    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch = sequences[start:start+batch_size]
            toks = tokenizer(batch, return_tensors="pt", padding=True, truncation=False)
            toks = {k: v.to(device) for k, v in toks.items()}

            out = model(**toks)
            # out.last_hidden_state: (B, T, H)
            h = out.last_hidden_state

            # Exclude special tokens if tokenizer adds them:
            # Typically: [CLS] seq [EOS] => we pool only positions corresponding to residues.
            # We'll use attention_mask to pool only "real tokens", then optionally remove special tokens.
            mask = toks["attention_mask"].unsqueeze(-1)  # (B, T, 1)
            h_masked = h * mask
            denom = mask.sum(dim=1).clamp(min=1.0)       # (B, 1)

            pooled = h_masked.sum(dim=1) / denom         # (B, H)
            all_embs.append(pooled.detach().cpu().numpy())

    return np.vstack(all_embs)

seqs_train = features_df_train["peptide"].tolist()
seqs_test = features_df_test["peptide"].tolist()
esm_emb_train = esm2_pooled_embeddings(seqs_train, batch_size=256)
esm_emb_test = esm2_pooled_embeddings(seqs_test, batch_size=256)

esm_dim_train = esm_emb_train.shape[1]
esm_dim_test = esm_emb_test.shape[1]

esm_train_cols = [f"esm2_{i}" for i in range(esm_dim_train)]
esm_test_cols = [f"esm2_{i}" for i in range(esm_dim_test)]
esm_test_df = pd.DataFrame(esm_emb_test, columns=esm_test_cols, index=features_df_test.index)
esm_train_df = pd.DataFrame(esm_emb_train, columns=esm_train_cols, index=features_df_train.index)

features_df_train = pd.concat([features_df_train, esm_train_df], axis=1)
features_df_test = pd.concat([features_df_test, esm_test_df], axis=1)
features_df_train.head()
features_df_test.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,pep_ID,peptide,total,wash1,wash2,wash3,wash4,vhse_pos1_d1,vhse_pos1_d2,vhse_pos1_d3,...,esm2_310,esm2_311,esm2_312,esm2_313,esm2_314,esm2_315,esm2_316,esm2_317,esm2_318,esm2_319
0,15519081,QVEVFYATARGH,255,0.588235,0.141176,0.003922,0.266667,-0.96,0.12,0.18,...,-0.009897,-0.009189,0.169641,0.010913,-0.025071,-0.000097,0.049135,0.201722,0.211197,-0.213723
1,18128100,SRDSSNSMSMGV,910,0.641758,0.098901,0.003297,0.256044,-0.67,-0.86,-1.07,...,-0.060535,-0.053442,-0.029418,0.285713,0.012824,0.284209,-0.054561,0.213732,0.166284,0.033343
2,19785195,TMQGNLVRSLQR,637,0.602826,0.105181,0.001570,0.290424,-0.34,-0.51,-0.55,...,-0.025580,-0.055434,0.245210,0.209779,-0.018257,0.223053,0.106339,0.150962,0.120834,-0.037394
3,11444486,LRFADVVFSQMD,209,0.483254,0.157895,0.009569,0.349282,1.36,0.07,0.26,...,-0.011713,0.039752,0.171744,0.030140,0.010936,0.033909,0.043093,0.140372,0.189610,-0.079325
4,153407,ADVDEQAVRKHP,1084,0.634686,0.095941,0.002768,0.266605,0.15,-1.11,-1.35,...,-0.045211,0.062396,0.172589,0.014480,-0.063036,-0.000398,0.056693,0.276056,0.102222,-0.131145


In [12]:
#Export fetures
features_df_train.to_parquet("peptide_features_train.parquet", index=False)
features_df_test.to_parquet("peptide_features_test.parquet", index=False)
print("Saved train features to:", Path("peptide_features_train.parquet").resolve())
print("Saved test features to:", Path("peptide_features_test.parquet").resolve())

Saved train features to: /content/peptide_features_train.parquet
Saved test features to: /content/peptide_features_test.parquet


# Task
Prepare training and testing data from `features_df_train` and `features_df_test` by creating feature matrices (`X_train`, `X_test`), target variables (`y_train`, `y_test` for 'wash1'), and sample weights (`sample_weights_train` from 'total'). Then, train a machine learning model (e.g., RandomForestRegressor or XGBoostRegressor) using the training data and evaluate its performance on the test data, reporting relevant metrics such as MAE, RMSE, or R-squared. Finally, summarize the model's training and evaluation results.

## Prepare Training Features and Target

### Subtask:
Create the feature matrix `X_train` by selecting relevant columns from `features_df_train`, excluding 'pep_ID', 'peptide', and 'wash1' through 'wash4'. Define the target variable `y_train` as 'wash1' from `features_df_train`. Also, extract 'total' from `features_df_train` to use as `sample_weights_train`.


**Reasoning**:
To prepare the training data, I will create a list of columns to be excluded from the feature matrix, then define X_train, y_train, and sample_weights_train based on the instructions provided.



In [13]:
exclude_cols = ['pep_ID', 'peptide', 'wash1', 'wash2', 'wash3', 'wash4', 'total']
X_train = features_df_train.drop(columns=exclude_cols)
y_train = features_df_train['wash1']
sample_weights_train = features_df_train['total']

print("X_train head:")
print(X_train.head())
print("\ny_train head:")
print(y_train.head())
print("\nsample_weights_train head:")
print(sample_weights_train.head())

X_train head:
   vhse_pos1_d1  vhse_pos1_d2  vhse_pos1_d3  vhse_pos1_d4  vhse_pos1_d5  \
0          0.15         -1.11         -1.35         -0.92          0.02   
1          0.15         -1.11         -1.35         -0.92          0.02   
2          0.15         -1.11         -1.35         -0.92          0.02   
3          0.15         -1.11         -1.35         -0.92          0.02   
4          0.15         -1.11         -1.35         -0.92          0.02   

   vhse_pos1_d6  vhse_pos1_d7  vhse_pos1_d8  vhse_pos2_d1  vhse_pos2_d2  ...  \
0         -0.91          0.36         -0.48          0.15         -1.11  ...   
1         -0.91          0.36         -0.48          0.15         -1.11  ...   
2         -0.91          0.36         -0.48          0.15         -1.11  ...   
3         -0.91          0.36         -0.48          0.15         -1.11  ...   
4         -0.91          0.36         -0.48          0.15         -1.11  ...   

   esm2_310  esm2_311  esm2_312  esm2_313  esm2_314  e

**Reasoning**:
Following the preparation of training data, the next step is to prepare the test data by creating the feature matrix `X_test`, target variable `y_test`, and sample weights `sample_weights_test` from `features_df_test`, excluding the specified columns.



In [14]:
exclude_cols = ['pep_ID', 'peptide', 'wash1', 'wash2', 'wash3', 'wash4', 'total']
X_test = features_df_test.drop(columns=exclude_cols)
y_test = features_df_test['wash1']
sample_weights_test = features_df_test['total']

print("X_test head:")
print(X_test.head())
print("\ny_test head:")
print(y_test.head())
print("\nsample_weights_test head:")
print(sample_weights_test.head())

X_test head:
   vhse_pos1_d1  vhse_pos1_d2  vhse_pos1_d3  vhse_pos1_d4  vhse_pos1_d5  \
0         -0.96          0.12          0.18          0.16          0.09   
1         -0.67         -0.86         -1.07         -0.41         -0.32   
2         -0.34         -0.51         -0.55         -1.06          0.01   
3          1.36          0.07          0.26         -0.80          0.22   
4          0.15         -1.11         -1.35         -0.92          0.02   

   vhse_pos1_d6  vhse_pos1_d7  vhse_pos1_d8  vhse_pos2_d1  vhse_pos2_d2  ...  \
0          0.42         -0.20         -0.41          0.76         -0.92  ...   
1          0.27         -0.64          0.11         -1.47          1.45  ...   
2         -0.01         -0.79          0.39          1.01         -0.53  ...   
3         -1.37          0.08         -0.62         -1.47          1.45  ...   
4         -0.91          0.36         -0.48         -1.15          0.67  ...   

   esm2_310  esm2_311  esm2_312  esm2_313  esm2_314  es

In [17]:
pip install xgboost

## Train the XGBoost Regressor Model

### Subtask:
Train an XGBoost Regressor model using the prepared `X_train`, `y_train`, and `sample_weights_train`. Set a random state for reproducibility.

#### Instructions
1. Import `XGBRegressor` from `xgboost`.
2. Initialize `XGBRegressor` with `random_state=42`.
3. Fit the model using `X_train`, `y_train`, and `sample_weights_train`.

**Reasoning**:
To train the model as instructed, I will import `XGBRegressor`, initialize it, and then fit it to the training data using the provided features, target, and sample weights.



In [23]:
from xgboost import XGBRegressor

# Initialize XGBoost Regressor with a random state for reproducibility
model = XGBRegressor(random_state=42)

# Train the model using X_train, y_train, and sample_weights_train
model.fit(X_train, y_train, sample_weight=sample_weights_train)

print("XGBoost Regressor model trained successfully.")

XGBoost Regressor model trained successfully.


## Evaluate Model Performance

### Subtask:
Make predictions on the test set (`X_test`) using the trained `model` and then calculate and report key regression metrics (MAE, RMSE, R-squared) comparing the predictions (`y_pred`) with the actual test targets (`y_test`).

#### Instructions
1. Use the trained `model` to make predictions on `X_test`, storing the results in `y_pred`.
2. Import necessary metrics: `mean_absolute_error`, `mean_squared_error`, and `r2_score` from `sklearn.metrics`.
3. Calculate the Mean Absolute Error (MAE) between `y_test` and `y_pred`.
4. Calculate the Root Mean Squared Error (RMSE) between `y_test` and `y_pred`.
5. Calculate the R-squared score between `y_test` and `y_pred`.
6. Print the calculated MAE, RMSE, and R-squared values.

**Reasoning**:
Now that the model is trained, I will make predictions on the test set and calculate the specified regression metrics (MAE, RMSE, R-squared) to evaluate its performance.



In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r_squared = r2_score(y_test, y_pred)

# Print the metrics
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2): {r_squared:.4f}")

Mean Absolute Error (MAE): 0.0793
Root Mean Squared Error (RMSE): 0.1108
R-squared (R2): 0.3510


## Summary of Model Training and Evaluation

The XGBoost Regressor model was successfully trained using the `X_train` features, `y_train` target ('wash1' values), and `sample_weights_train` (derived from 'total' column, ensuring all values were positive).

Upon evaluation on the test set (`X_test` and `y_test`), the model achieved the following performance metrics:

- **Mean Absolute Error (MAE):** 0.0793
- **Root Mean Squared Error (RMSE):** 0.1108
- **R-squared (R2):** 0.3510

These metrics indicate that the model provides a reasonable fit to the data, with an MAE of approximately 0.08, meaning predictions are, on average, 0.08 units away from the actual 'wash1' values. The R-squared value of 0.3510 suggests that about 35.1% of the variance in the 'wash1' values can be explained by the model's features.

## Final Task

### Subtask:
Summarize the model's training and evaluation results, highlighting the performance of Model 1 and any insights gained from the feature preparation based on the user's instructions.


## Summary:

### Data Analysis Key Findings

*   **Feature and Target Preparation**: Training and testing feature matrices (`X_train`, `X_test`) were successfully created by excluding specific columns (`pep_ID`, `peptide`, `wash1` through `wash4`, `total`). The target variables (`y_train`, `y_test`) were defined as the 'wash1' column.
*   **Sample Weight Adjustment**: Sample weights for both training (`sample_weights_train`) and testing (`sample_weights_test`) were derived from the 'total' column. All non-positive values in these weights were replaced with a small positive number ($1e-6$) to ensure proper model training.
*   **Model Training**: An XGBoost Regressor model was successfully trained using `X_train`, `y_train`, and the adjusted `sample_weights_train`.
*   **Model Performance (Model 1)**:
    *   **Mean Absolute Error (MAE)**: The model achieved an MAE of 0.0793, indicating that, on average, predictions deviated from actual 'wash1' values by approximately 0.08 units.
    *   **Root Mean Squared Error (RMSE)**: The RMSE was 0.1108.
    *   **R-squared (R2)**: The R-squared value was 0.3510, suggesting that approximately 35.1% of the variance in the 'wash1' values could be explained by the features used in the model.

### Insights or Next Steps

*   The current model explains a moderate portion of the variance in the target variable, indicating potential for improvement. Further feature engineering or exploration of more complex model architectures could enhance predictive power.
*   The use of sample weights suggests that certain data points might be more important. Analyzing the distribution of these weights and their impact on prediction errors could reveal areas for targeted model refinement.
